# Python bridge for 02a

## A short code refresher

Use this optional notebook alongside 02a if you would like a closer look at the Python mechanics behind a permutation test. It takes about 15–20 minutes and uses a small example whose rows can be inspected directly.

The 02a lecture develops the statistical reasoning. This companion focuses on five code patterns:

1. turn a comparison into a Boolean rate;
2. convert pandas objects to NumPy arrays;
3. use a seeded random-number generator to permute labels;
4. select values with `.loc`; and
5. pass a statistic function to SciPy and inspect the returned object.

## Create a small table

Each row has a recorded group label and one numerical value. The short table lets you see what changes when Python reassigns the labels.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import permutation_test

In [ ]:
records = pd.DataFrame(
    {
        "group": ["A"] * 7 + ["B"] * 7,
        "value": [8, 10, 11, 14, 16, 19, 23, 5, 7, 9, 10, 12, 13, 15],
    }
)

records

## Turn a comparison into a rate

The `.gt(12)` expression asks whether each value is greater than 12. It returns `True` or `False` for every row.

In [ ]:
records["above_12"] = records["value"].gt(12)

records

pandas treats `True` as 1 and `False` as 0 when it calculates a mean. The mean of a Boolean column is therefore the proportion of rows for which the condition is true.

In [ ]:
records.groupby("group")["above_12"].mean()

This is the mechanism behind an exceedance rate. Changing `12` changes the threshold represented by the Boolean column.

## Convert columns to arrays

`records["group"]` is a pandas `Series`. The `.to_numpy()` method extracts its values as a NumPy array, which is the input form used by the random-number generator and many SciPy functions.

In [ ]:
labels = records["group"].to_numpy()
values = records["value"].to_numpy()

print(type(labels))
print("label shape:", labels.shape)
print("value shape:", values.shape)

Both arrays have shape `(14,)`: each is a one-dimensional sequence with one entry for every row. Their positions still correspond, so `labels[0]` and `values[0]` came from the same record.

## Permute the labels

`np.random.default_rng(7130)` creates a random-number generator. The seed `7130` makes the sequence reproducible when the notebook is restarted and run from the beginning.

`rng.permutation(labels)` returns a new array with the same labels in a random order. It does not change the original `labels` array.

In [ ]:
rng = np.random.default_rng(7130)
reassigned_labels = rng.permutation(labels)

pd.DataFrame(
    {
        "recorded label": labels,
        "reassigned label": reassigned_labels,
        "unchanged value": values,
    }
)

Because permutation only changes order, the number of `A` and `B` labels remains the same.

In [ ]:
pd.DataFrame(
    {
        "recorded": pd.Series(labels).value_counts(),
        "reassigned": pd.Series(reassigned_labels).value_counts(),
    }
).sort_index()

## Select the two arrays

The `.loc[rows, column]` pattern selects values using a Boolean row condition and a column name. The following code first attaches the reassigned labels, then creates one numerical array for each reassigned group.

In [ ]:
reassigned_records = records.assign(reassigned_group=reassigned_labels)

reassigned_a = reassigned_records.loc[
    reassigned_records["reassigned_group"] == "A", "value"
].to_numpy()
reassigned_b = reassigned_records.loc[
    reassigned_records["reassigned_group"] == "B", "value"
].to_numpy()

reassigned_a, reassigned_b

The row condition appears before the comma. The requested column appears after it. Calling `.to_numpy()` at the end converts each selected `Series` to the array expected by the statistic function.

## Define one statistic function

SciPy needs a function that accepts two arrays and returns one number. This function returns the median of its first input minus the median of its second input.

In [ ]:
def median_difference(x, y):
    return np.median(x) - np.median(y)

Use `.loc` to create the two recorded groups, then call the function directly. The order of the inputs controls the sign of the result.

In [ ]:
recorded_a = records.loc[records["group"] == "A", "value"].to_numpy()
recorded_b = records.loc[records["group"] == "B", "value"].to_numpy()

print("A minus B:", median_difference(recorded_a, recorded_b))
print("B minus A:", median_difference(recorded_b, recorded_a))

## Inspect the SciPy result

`permutation_test` repeatedly reassigns values between the two groups and calls `median_difference`. The result is an object containing several related outputs rather than one number.

In [ ]:
result = permutation_test(
    (recorded_a, recorded_b),
    statistic=median_difference,
    permutation_type="independent",
    alternative="two-sided",
    n_resamples=999,
    rng=np.random.default_rng(7130),
)

print("observed statistic:", result.statistic)
print("p-value:", result.pvalue)
print("null-distribution shape:", result.null_distribution.shape)

The attributes have different roles:

- `result.statistic` is the statistic calculated from the recorded input order;
- `result.pvalue` is SciPy's comparison of that statistic with the permutation distribution; and
- `result.null_distribution` is the array of statistics calculated after reassignment.

## Try one small modification

The callable supplied through `statistic=` determines what SciPy recalculates. Replace the median with the mean, then compare the returned statistic and permutation distribution.

In [ ]:
def mean_difference(x, y):
    return np.mean(x) - np.mean(y)


mean_result = permutation_test(
    (recorded_a, recorded_b),
    statistic=mean_difference,
    permutation_type="independent",
    alternative="two-sided",
    n_resamples=999,
    rng=np.random.default_rng(7130),
)

print("median difference:", result.statistic)
print("mean difference:", mean_result.statistic)

The SciPy call has the same structure. Passing a different function changes the quantity calculated for the recorded data and every reassignment.

## Ready for 02a

You are ready to return to 02a when you can recognize these patterns:

```text
series.gt(threshold)                         # compare every value with a threshold
series.to_numpy()                            # extract values as an array
rng.permutation(labels)                      # return the labels in a new order
frame.loc[row_condition, "column"]          # select one column from matching rows
permutation_test(data, statistic=function)   # recalculate a function after reassignment
result.null_distribution                     # inspect one field of the returned object
```

The important implementation checks are that the labels and values still align by position, the statistic's input order matches the intended direction, and the returned field is the one needed for the next step.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).